# Automatic real-session dataset: silver labels

No manual labeling is required for this path. Use a private copy of this notebook under `data/private/` because its evidence viewer can display conversation text. The builder infers labels from request intent, actual final responses and recognized specialist delegation. Root and delegated model names, tool requests/starts and failed tool calls are recorded as observations, not proof of correctness or optimal model selection.

This snapshot has 15 non-development records: 8 fallback, 6 code generation and 1 debugging. Eleven have corroborating weak signals; four are tentative. Nine turns from the routing experiment remain excluded. These are **unreviewed silver labels**, not human-validated gold or a balanced six-route benchmark. Any measured score is agreement with this automatic labeler.

Post-decision responses and telemetry are stored in `evidence.jsonl`, separately from the router input in `silver-records.jsonl`. The latter retains only the original sanitized query and earlier context. Missing attachments remain missing. Unknown cases abstain rather than being silently labeled fallback. The builder makes no model calls and grants no permission to transmit private data.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'contracts' / 'agents.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook inside the routing experiment project')
sys.path.insert(0, str(ROOT))
from routing_lab.silver_data import load_silver_records
from routing_lab.data import load_contracts
from routing_lab.backends import RulesBackend
from routing_lab.endpoint import ClefEndpointBackend
from routing_lab.evaluation import run_experiment, sha256
from routing_lab.prompt_experiment import expanded_contracts

SILVER_DIR = ROOT / 'data' / 'private' / 'session-silver-20260930-111833'
silver_manifest = json.loads((SILVER_DIR / 'manifest.json').read_text(encoding='utf-8'))
silver_records = load_silver_records(SILVER_DIR)
if sha256(SILVER_DIR / 'evidence.jsonl') != silver_manifest['files']['evidence.jsonl']:
    raise ValueError('Outcome evidence changed since labeling')
silver_evidence = [json.loads(line) for line in (SILVER_DIR / 'evidence.jsonl').read_text(encoding='utf-8').split('\n') if line.strip()]
keys = ['labeler_version', 'candidate_count', 'eligible_candidate_count', 'silver_record_count', 'label_coverage', 'route_counts', 'eligible_label_status_counts', 'recorded_root_models', 'recorded_delegated_agents', 'remote_calls']
print(json.dumps({key: silver_manifest[key] for key in keys}, indent=2))

## Inspect one automatically labeled example

The first display below is the router's input. The second is **labeling-only evidence** from afterward; never concatenate it into `state` or use it to construct a prompt. `ranking_scores_not_probabilities` are deterministic rule weights, not calibrated confidence. A tool reporting success and an assistant saying 'fixed' are not independent task verification.

In [ ]:
SHOW_PRIVATE_EXAMPLE = False
SILVER_EXAMPLE_INDEX = 0
if SHOW_PRIVATE_EXAMPLE:
    silver_example = silver_records[SILVER_EXAMPLE_INDEX]
    evidence = next(row for row in silver_evidence if row['id'] == silver_example.id)
    print('ROUTER INPUT ONLY:')
    print(json.dumps({'query': silver_example.query, 'context': silver_example.context}, indent=2))
    print('\nAUTOMATIC LABEL AND POST-DECISION EVIDENCE (never router input):')
    print(json.dumps(evidence, indent=2))
else:
    print('Private text hidden. Enable SHOW_PRIVATE_EXAMPLE to inspect it locally.')

## Local end-to-end smoke test

The rules baseline uses no network or model weights. It checks that this dataset works with the existing evaluator. Since both the labeler and this baseline use lexical heuristics, their agreement is not independent quality evidence. A saved smoke result is displayed by default; rerunning creates a new folder.

In [ ]:
RUN_LOCAL_SILVER_SMOKE = False
SILVER_SMOKE_DIR = ROOT / 'outputs' / 'silver-rules-smoke-20260930'
if RUN_LOCAL_SILVER_SMOKE:
    SILVER_SMOKE_DIR = ROOT / 'outputs' / f'silver-rules-smoke-{uuid4().hex[:10]}'
    run_experiment(RulesBackend(), silver_records, SILVER_SMOKE_DIR, probe=True,
        source_paths={'dataset': SILVER_DIR / 'silver-records.jsonl', 'silver_manifest': SILVER_DIR / 'manifest.json'})
if (SILVER_SMOKE_DIR / 'summary.json').exists():
    smoke = json.loads((SILVER_SMOKE_DIR / 'summary.json').read_text(encoding='utf-8'))
    print('Pipeline smoke test: agreement with automatic labels, NOT verified routing accuracy.')
    print(json.dumps({key: smoke[key] for key in ['mode', 'count', 'strict_accuracy', 'macro_f1']}, indent=2))
else:
    print('No saved smoke result; enable RUN_LOCAL_SILVER_SMOKE to create one locally.')

## Optional Clef endpoint probe

Manual labels are not required. A separate, explicit **batch privacy approval** is still required before transmitting real conversations. Heuristic redaction is not a guarantee that all sensitive information has been removed. Only enable the switches below if you have permission to send these sanitized inputs to your endpoint.

Run each fixed criterion version separately on the same snapshot. Interpret scores as silver-label agreement, retaining tentative cases and coverage statistics. Do not claim these scores measure verified correctness, optimal model choice, or justify fine-tuning. The automatic labeler and the model may share errors.

In [ ]:
RUN_CLEF_SILVER_PROBE = False
ALLOW_PRIVATE_REMOTE = False
SILVER_VARIANT = 'baseline'
if RUN_CLEF_SILVER_PROBE:
    if not ALLOW_PRIVATE_REMOTE:
        raise RuntimeError('Explicit batch permission to transmit private inputs is required')
    if SILVER_VARIANT not in {'baseline', 'expanded_v1'}:
        raise ValueError('Choose baseline or expanded_v1')
    silver_records = load_silver_records(SILVER_DIR)
    silver_contracts = load_contracts(ROOT / 'contracts' / 'agents.json')
    if SILVER_VARIANT == 'expanded_v1':
        silver_contracts = expanded_contracts(silver_contracts, ROOT / 'contracts' / 'criteria-expanded-v1.json')
    silver_connection = json.loads((ROOT / 'azure' / 'online' / 'connection.json').read_text())
    silver_backend = ClefEndpointBackend(silver_contracts, url=silver_connection['scoring_uri'], context_mode='context')
    silver_output = ROOT / 'outputs' / f'clef-silver-{SILVER_VARIANT}-{uuid4().hex[:10]}'
    silver_result = run_experiment(silver_backend, silver_records, silver_output, probe=True,
        source_paths={'dataset': SILVER_DIR / 'silver-records.jsonl', 'silver_manifest': SILVER_DIR / 'manifest.json',
                      'contracts': ROOT / 'contracts' / 'agents.json', 'expanded_criteria': ROOT / 'contracts' / 'criteria-expanded-v1.json'})
    print('Agreement with automatically inferred labels; not a gold benchmark:', silver_output)
    print('Label coverage:', silver_manifest['label_coverage'], '| Label status:', silver_manifest['eligible_label_status_counts'])
    print(json.dumps(silver_result, indent=2))
else:
    print('No private session inputs sent to Azure. Both remote switches remain disabled.')